# 01.1 — PyTorch Baselines on Test Set (ImageNet Val)

Same sweep as `01_fp32_baselines_sweep` but evaluated on the ImageNet validation split (test set).

Results saved under `resultsv2/test_runs/`.

In [9]:

SPLIT = "val"              # ImageNet val split (= "test" for this project)
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/test_infer"

In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [11]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from src.config import ExperimentConfig, with_overrides, set_seed
from src.data import get_dataloader
from src.model import get_model
from src.eval import evaluate
from utils.precision import apply_precision
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [12]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
INPUT_BITS = [8, 4, 2, 1]

checkpoints = {}
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if seed_dir.is_dir() and (seed_dir / "best.pth").exists():
            checkpoints[(bits, seed_dir.name)] = str(seed_dir / "best.pth")

print("Checkpoints found:")
for (bits, seed), path in checkpoints.items():
    print(f"  {bits}bit / {seed}: {path}")

Checkpoints found:
  8bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_1/best.pth
  8bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_2/best.pth
  8bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth
  4bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_1/best.pth
  4bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_2/best.pth
  4bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_42/best.pth
  2bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_1/best.pth
  2bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_2/best.pth
  2bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_42/best.pth
  1bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_1bit/seed_1/best

In [13]:
def run_test_experiment(cfg, checkpoint_path, split=SPLIT):
    cfg = cfg.normalized()
    cfg.validate()
    set_seed(cfg)

    criterion = nn.CrossEntropyLoss()
    loader = get_dataloader(cfg, split=split)

    model = apply_precision(get_model(cfg, checkpoint_path=checkpoint_path), cfg)
    t0 = time.perf_counter()
    tracker = evaluate(model, loader, cfg, criterion=criterion)
    elapsed = round(time.perf_counter() - t0, 3)

    payload = {
        "status": "ok",
        "run_id": cfg.run_id(),
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "results": tracker.summary(),
        "total_eval_time_sec": elapsed,
        "split": split,
        "error": None,
    }

    out_path = Path(cfg.output_root) / cfg.run_id() / "result.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    print(f"  Saved: {out_path}")

    return payload

## FP32 Full Sweep (b=8, 4, 2, 1)

In [14]:
fp32_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    print(f"\n{'='*60}")
    print(f"  {bits}bit / {seed}  |  precision: fp32")
    print(f"{'='*60}")

    cfg = with_overrides(
        ExperimentConfig(
            backend="pytorch", device="cuda", batch_size=1,
            seed=42, num_eval_batches=None,
            output_root=f"{OUTPUT_ROOT}/fp32_{bits}bit/{seed}",
        ),
        model_precision="fp32", input_quant_bits=bits,
    )
    payload = run_test_experiment(cfg, checkpoint_path=ckpt_path)
    fp32_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")


  8bit / seed_1  |  precision: fp32
[data] Filtered to 5000 samples across 100 classes.
Evaluating on 5000 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 2.79 ms/batch
  Batch [50/5000] Top-1: 95.00% | Top-5: 100.00% | Infer: 2.77 ms/batch
  Batch [60/5000] Top-1: 93.33% | Top-5: 100.00% | Infer: 2.66 ms/batch
  Batch [70/5000] Top-1: 90.00% | Top-5: 97.50% | Infer: 2.80 ms/batch
  Batch [80/5000] Top-1: 88.00% | Top-5: 98.00% | Infer: 2.71 ms/batch
  Batch [90/5000] Top-1: 90.00% | Top-5: 98.33% | Infer: 2.74 ms/batch
  Batch [100/5000] Top-1: 88.57% | Top-5: 98.57% | Infer: 2.76 ms/batch
  Batch [110/5000] Top-1: 88.75% | Top-5: 98.75% | Infer: 2.81 ms/batch
  Batch [120/5000] Top-1: 87.78% | Top-5: 97.78% | Infer: 2.91 ms/batch
  Batch [130/5000] Top-1: 89.00% | Top-5: 98.00% | Infer: 2.95 ms/batch
  Batch [140/5000] Top-1: 89.09% | Top-5: 98.18% | Infer: 2.90 ms/batch
  

## FP16 Full Sweep (b=8, 4, 2, 1)

In [15]:
fp16_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    print(f"\n{'='*60}")
    print(f"  {bits}bit / {seed}  |  precision: fp16")
    print(f"{'='*60}")

    cfg = with_overrides(
        ExperimentConfig(
            backend="pytorch", device="cuda", batch_size=1,
            seed=42, num_eval_batches=None,
            output_root=f"{OUTPUT_ROOT}/fp32_{bits}bit/{seed}",
        ),
        model_precision="fp16", input_quant_bits=bits,
    )
    payload = run_test_experiment(cfg, checkpoint_path=ckpt_path)
    fp16_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")


  8bit / seed_1  |  precision: fp16
[data] Filtered to 5000 samples across 100 classes.
Evaluating on 5000 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 2.75 ms/batch
  Batch [50/5000] Top-1: 95.00% | Top-5: 100.00% | Infer: 2.64 ms/batch
  Batch [60/5000] Top-1: 93.33% | Top-5: 100.00% | Infer: 2.67 ms/batch
  Batch [70/5000] Top-1: 90.00% | Top-5: 97.50% | Infer: 2.71 ms/batch
  Batch [80/5000] Top-1: 88.00% | Top-5: 98.00% | Infer: 2.78 ms/batch
  Batch [90/5000] Top-1: 90.00% | Top-5: 98.33% | Infer: 2.74 ms/batch
  Batch [100/5000] Top-1: 88.57% | Top-5: 98.57% | Infer: 2.73 ms/batch
  Batch [110/5000] Top-1: 88.75% | Top-5: 98.75% | Infer: 2.67 ms/batch
  Batch [120/5000] Top-1: 87.78% | Top-5: 97.78% | Infer: 2.64 ms/batch
  Batch [130/5000] Top-1: 89.00% | Top-5: 98.00% | Infer: 2.64 ms/batch
  Batch [140/5000] Top-1: 89.09% | Top-5: 98.18% | Infer: 2.70 ms/batch
  

## Results Summary

In [16]:
all_rows = []
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        run_dir = f"{OUTPUT_ROOT}/fp32_{bits}bit/{seed_dir.name}"
        runs = load_runs(run_dir, status="ok")
        for r in flatten_runs(runs):
            r["seed"] = seed_dir.name
            r["input_bits_trained"] = bits
            all_rows.append(r)

df = pd.DataFrame(all_rows)

display_cols = [
    "seed", "input_bits_trained", "run_id",
    "cfg.model_precision", "cfg.input_quant_bits",
    "res.top1_acc", "res.top5_acc",
    "res.infer_ms_avg", "res.throughput_infer_sps", "res.total_samples",
]

df[display_cols].sort_values(
    ["input_bits_trained", "seed", "cfg.model_precision", "cfg.input_quant_bits"],
    ascending=[False, True, True, False],
)

,seed,input_bits_trained,run_id,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
0,seed_1,8,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,8,77.867203,93.380282,3.042169,328.712801,4970
1,seed_1,8,resnet18_pytorch_fp32_in8b_cuda_bs1,fp32,8,77.847082,93.400402,3.073514,325.360470,4970
2,seed_2,8,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,8,77.987928,93.843058,3.101329,322.442400,4970
3,seed_2,8,resnet18_pytorch_fp32_in8b_cuda_bs1,fp32,8,78.008048,93.843058,2.968521,336.868108,4970
4,seed_42,8,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,8,78.068410,93.521127,3.001022,333.219810,4970
5,seed_42,8,resnet18_pytorch_fp32_in8b_cuda_bs1,fp32,8,78.108652,93.501006,3.086705,323.970087,4970
6,seed_1,4,resnet18_pytorch_fp16_in4b_cuda_bs1,fp16,4,78.410463,93.641851,3.053762,327.464955,4970
7,seed_1,4,resnet18_pytorch_fp32_in4b_cuda_bs1,fp32,4,78.430584,93.682093,3.102357,322.335551,4970
8,seed_2,4,resnet18_pytorch_fp16_in4b_cuda_bs1,fp16,4,77.927565,93.742455,3.104329,322.130790,4970
9,seed_2,4,resnet18_pytorch_fp32_in4b_cuda_bs1,fp32,4,77.967807,93.742455,3.077610,324.927476,4970


In [17]:
avg_df = df[df["cfg.backend"] == "pytorch"].groupby(
    ["cfg.backend", "cfg.model_precision", "input_bits_trained"]
).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["cfg.model_precision", "input_bits_trained"],
    ascending=[True, True],
).reset_index(drop=True)
avg_df

,cfg.backend,cfg.model_precision,input_bits_trained,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean
0,pytorch,fp16,1,69.551,0.475,89.477,0.126,3.065,0.058,326.331
1,pytorch,fp16,2,76.888,0.425,93.139,0.053,3.075,0.046,325.244
2,pytorch,fp16,4,78.330,0.369,93.642,0.101,3.087,0.029,323.959
3,pytorch,fp16,8,77.975,0.101,93.581,0.237,3.048,0.050,328.125
4,pytorch,fp32,1,69.551,0.466,89.470,0.129,2.980,0.013,335.550
5,pytorch,fp32,2,76.861,0.413,93.139,0.053,2.984,0.034,335.128
6,pytorch,fp32,4,78.337,0.332,93.655,0.103,3.082,0.019,324.479
7,pytorch,fp32,8,77.988,0.132,93.581,0.232,3.043,0.065,328.733


In [18]:
bench_path = PROJECT_ROOT / "results" / "latency_bench" / "pytorch_fp32_in8b_cuda_bs1.json"
with open(bench_path) as f:
    bench = json.load(f)
bench_std = np.std(bench["latencies_ms"], ddof=1)

avg_df[["cfg.model_precision", "input_bits_trained",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {bench_std:.3f}", axis=1),
)[["cfg.model_precision", "input_bits_trained", "top1", "top5", "infer_ms"]]

,cfg.model_precision,input_bits_trained,top1,top5,infer_ms
0,fp16,1,69.55 ± 0.47,89.48 ± 0.13,3.065 ± 1.244
1,fp16,2,76.89 ± 0.42,93.14 ± 0.05,3.075 ± 1.244
2,fp16,4,78.33 ± 0.37,93.64 ± 0.10,3.087 ± 1.244
3,fp16,8,77.97 ± 0.10,93.58 ± 0.24,3.048 ± 1.244
4,fp32,1,69.55 ± 0.47,89.47 ± 0.13,2.980 ± 1.244
5,fp32,2,76.86 ± 0.41,93.14 ± 0.05,2.984 ± 1.244
6,fp32,4,78.34 ± 0.33,93.66 ± 0.10,3.082 ± 1.244
7,fp32,8,77.99 ± 0.13,93.58 ± 0.23,3.043 ± 1.244


In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "pytorch_avg_results_test.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")